# blocking-eval-v1
Generated from `code/business_entity_resolution/src/run_blocking_eval.py` by `jobs/build_notebook.py`. Edit the script, not this notebook.

In [ ]:
!pip install -q polars==1.44.2 rapidfuzz==3.14.6 anyascii==0.3.3 scipy==1.18.1 scikit-learn==1.9.1 sparse-dot-topn==1.2.0

In [ ]:
import os
os.makedirs('/tmp/src', exist_ok=True)
for root, dirs, files in os.walk('/kaggle/input'):
    print(root, dirs, files)

In [ ]:
%%writefile /tmp/src/blocking.py
"""Candidate generation (blocking): union of several searches, per country.

Methods (all per country; the country is a compute split, 0 true pairs cross countries):
- name       forward TF-IDF top-k on core_name
- name_city  forward TF-IDF top-k on core_name + city
- name_addr  forward TF-IDF top-k on core_name + normalised address
- reverse    for every S2/S3 record, its top-m S1 on core_name + address (all S1 of the split compete)
- rare       S2/S3 records sharing a rare core_name token (document frequency <= RARE_DF in the country)

TF-IDF: char_wb 3-4-grams, fitted per country on train + test records (no labels; a seeded sample
of up to FIT_SAMPLE texts). N-grams in more than MAX_DF of the texts are dropped: they carry almost no
weight and dominate the cost of the sparse product.

State buckets (compute split, not a filter): a forward query searches the pool records of its own
state plus all pool records without a state; an S1 without a state searches the whole country. The
reverse search runs within states only (records without a state are reached by the forward searches).
The rare-token index ignores states, so it recovers pairs whose states disagree.

Near-identical decoys are NOT filtered here; every method keeps its full top-k.
"""
from __future__ import annotations

import time
from collections.abc import Callable

import numpy as np
import polars as pl
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer
from sparse_dot_topn import sp_matmul_topn

SEED = 42
FIT_SAMPLE = 3_000_000
MAX_DF = 0.02
RARE_DF = 20
NGRAMS = (3, 4)
FORWARD = ("name", "name_city", "name_addr")
REVERSE_REP = "name_addr"
NONE = "__none__"
N_THREADS = 4


def texts(df: pl.DataFrame) -> pl.DataFrame:
    """Text representations used by the searches (all from norm-v3 columns)."""
    core = pl.col("core_name")
    return df.select(
        core.alias("name"),
        pl.concat_str([core, pl.col("city").fill_null("")], separator=" ").str.strip_chars().alias("name_city"),
        pl.concat_str([core, pl.col("addr_norm").str.replace_all(",", "")], separator=" ").str.strip_chars().alias("name_addr"),
    )


def fit_vectorizer(corpus: pl.Series) -> TfidfVectorizer:
    if corpus.len() > FIT_SAMPLE:
        corpus = corpus.sample(FIT_SAMPLE, seed=SEED)
    v = TfidfVectorizer(analyzer="char_wb", ngram_range=NGRAMS, min_df=2, max_df=MAX_DF,
                        sublinear_tf=True, dtype=np.float32)
    v.fit(corpus.to_list())
    return v


def _topn(Q: sparse.csr_matrix, P: sparse.csr_matrix, k: int) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Top-k columns of Q @ P.T per row: (row, col, score, rank 1..k)."""
    if Q.shape[0] == 0 or P.shape[0] == 0:
        e = np.empty(0, dtype=np.int64)
        return e, e, np.empty(0, dtype=np.float32), e
    R = sp_matmul_topn(Q, P.T.tocsr(), top_n=min(k, P.shape[0]), threshold=0.0, sort=True, n_threads=N_THREADS)
    R = R.tocsr()
    counts = np.diff(R.indptr)
    rows = np.repeat(np.arange(R.shape[0]), counts)
    ranks = np.arange(R.nnz) - np.repeat(R.indptr[:-1], counts) + 1
    return rows, R.indices.astype(np.int64), R.data.astype(np.float32), ranks


def forward(Q, q_bucket: np.ndarray, P, p_bucket: np.ndarray, k: int) -> pl.DataFrame:
    """For each query row: top-k pool rows among its state bucket + pool rows without a state."""
    out = []
    p_none = np.flatnonzero(p_bucket == NONE)
    for b in np.unique(q_bucket):
        qi = np.flatnonzero(q_bucket == b)
        pi = np.arange(P.shape[0]) if b == NONE else np.concatenate([np.flatnonzero(p_bucket == b), p_none])
        r, c, s, rk = _topn(Q[qi], P[pi], k)
        out.append(pl.DataFrame({"qi": qi[r], "pi": pi[c], "score": s, "rank": rk}))
    return pl.concat(out) if out else pl.DataFrame(schema={"qi": pl.Int64, "pi": pl.Int64, "score": pl.Float32, "rank": pl.Int64})


def reverse(S, s_bucket: np.ndarray, P, p_bucket: np.ndarray, m: int) -> pl.DataFrame:
    """For each pool row with a state: its top-m S1 rows in the same state. Returns (qi = S1 row, pi, score, rank)."""
    out = []
    for b in np.unique(p_bucket):
        if b == NONE:
            continue
        pi = np.flatnonzero(p_bucket == b)
        si = np.flatnonzero(s_bucket == b)
        r, c, s, rk = _topn(P[pi], S[si], m)
        out.append(pl.DataFrame({"qi": si[c], "pi": pi[r], "score": s, "rank": rk}))
    return pl.concat(out) if out else pl.DataFrame(schema={"qi": pl.Int64, "pi": pl.Int64, "score": pl.Float32, "rank": pl.Int64})


def rare_tokens(q_names: pl.Series, p_names: pl.Series, df_counts: pl.DataFrame, max_df: int = RARE_DF) -> pl.DataFrame:
    """Pool rows sharing a rare core_name token with the query (token df <= max_df in the country).
    score = number of shared rare tokens; rank by score (ties by pool order)."""
    rare = df_counts.filter((pl.col("df") <= max_df) & (pl.col("t").str.len_chars() >= 3)).select("t")
    tok = lambda s, name: (pl.DataFrame({name: np.arange(s.len()), "t": s.str.split(" ")})
                           .explode("t", empty_as_null=True).filter(pl.col("t").is_not_null()).unique()
                           .join(rare, on="t"))
    qt, pt = tok(q_names, "qi"), tok(p_names, "pi")
    hits = qt.join(pt, on="t").group_by("qi", "pi").agg(pl.len().cast(pl.Float32).alias("score"))
    return hits.with_columns(
        pl.col("score").rank("ordinal", descending=True).over("qi").cast(pl.Int64).alias("rank")
    ).select("qi", "pi", "score", "rank")


def token_df(names: pl.Series) -> pl.DataFrame:
    """Document frequency of core_name tokens (distinct per record)."""
    return (pl.DataFrame({"i": np.arange(names.len()), "t": names.str.split(" ")})
            .explode("t", empty_as_null=True).filter(pl.col("t").is_not_null() & (pl.col("t") != "")).unique()
            .group_by("t").agg(pl.len().alias("df")))


def block_country(s1: pl.DataFrame, pool: pl.DataFrame, query_mask: np.ndarray, fit_corpus: dict[str, pl.Series],
                  df_counts: pl.DataFrame, k: int, m: int, log: Callable[[str], None] = print) -> tuple[pl.DataFrame, dict]:
    """All methods for one country. s1: ALL S1 of the split (reverse competition); query_mask: S1 rows to
    generate candidates for. Returns long table (s1_row, pool_row, method, score, rank) and timings."""
    ts, tp = texts(s1), texts(pool)
    s_bucket = s1["state"].fill_null(NONE).to_numpy()
    p_bucket = pool["state"].fill_null(NONE).to_numpy()
    qrows = np.flatnonzero(query_mask)
    parts, timing = [], {}
    for rep in FORWARD:
        t0 = time.time()
        v = fit_vectorizer(fit_corpus[rep])
        S, P = v.transform(ts[rep].to_list()), v.transform(tp[rep].to_list())
        t1 = time.time()
        f = forward(S[qrows], s_bucket[qrows], P, p_bucket, k)
        f = f.with_columns(pl.Series("qi", qrows[f["qi"].to_numpy()]))  # query-local row -> S1 row
        parts.append(f.with_columns(pl.lit(rep).alias("method")))
        t2 = time.time()
        timing[rep] = {"fit_transform_s": t1 - t0, "search_s": t2 - t1, "queries": len(qrows), "pool": P.shape[0],
                       "pairs_evaluated": _pairs_evaluated(s_bucket[qrows], p_bucket)}
        log(f"  {rep}: fit+transform {t1 - t0:.0f}s, forward search {t2 - t1:.0f}s")
        if rep == REVERSE_REP:
            t3 = time.time()
            r = reverse(S, s_bucket, P, p_bucket, m)
            r = r.filter(pl.Series(query_mask[r["qi"].to_numpy()], dtype=pl.Boolean))  # keep query S1 only
            parts.append(r.with_columns(pl.lit("reverse").alias("method")))
            t4 = time.time()
            timing["reverse"] = {"search_s": t4 - t3, "queries": int((p_bucket != NONE).sum()), "pool": S.shape[0],
                                 "pairs_evaluated": _pairs_evaluated(p_bucket[p_bucket != NONE], s_bucket, with_none=False)}
            log(f"  reverse: search {t4 - t3:.0f}s")
        del S, P
    t5 = time.time()
    rr = rare_tokens(s1["core_name"][qrows], pool["core_name"], df_counts)
    rr = rr.with_columns(pl.Series("qi", qrows[rr["qi"].to_numpy()], dtype=pl.Int64))
    parts.append(rr.with_columns(pl.lit("rare").alias("method")))
    timing["rare"] = {"search_s": time.time() - t5}
    log(f"  rare: {time.time() - t5:.0f}s")
    cols = ["qi", "pi", "method", "score", "rank"]
    return pl.concat([p.select(cols) for p in parts]), timing


def _pairs_evaluated(q_bucket: np.ndarray, p_bucket: np.ndarray, with_none: bool = True) -> float:
    """Number of (query, pool) similarity evaluations implied by the bucket sizes (cost model)."""
    pb = pl.Series(p_bucket).value_counts()
    sizes = dict(zip(pb[pb.columns[0]].to_list(), pb["count"].to_list()))
    n_none = sizes.get(NONE, 0) if with_none else 0
    total = 0.0
    for b, n in zip(*np.unique(q_bucket, return_counts=True)):
        total += n * (len(p_bucket) if b == NONE else sizes.get(b, 0) + n_none)
    return total


In [ ]:
import sys
sys.path.insert(0, '/tmp/src')
sys.argv = ['run_blocking_eval.py', '--norm-dir', '/kaggle/input', '--data-dir', '/kaggle/input', '--split', '/kaggle/input', '--out-dir', '/kaggle/working']

In [ ]:
"""Blocking evaluation on the validation split against the FULL train S2/S3 pool of each country.

Usage: python src/run_blocking_eval.py --norm-dir artifacts/normalised --data-dir data_parquet \
           --split outputs/eda/g25_split.parquet --out-dir artifacts/blocking_eval
--norm-dir / --split may be folders to search (e.g. /kaggle/input). Outputs in --out-dir:
  val_candidates.parquet  one row per (S1, candidate): rank and score per method (null = not found)
  recall_by_method.csv, recall_union_grid.csv, recall_groups.csv, timing.csv, test_estimate.csv, blocking_report.md
"""
import argparse
import json
import resource
import sys
import time
from pathlib import Path

import numpy as np
import polars as pl
from rapidfuzz import fuzz, process

try:
    sys.path.insert(0, str(Path(__file__).resolve().parent))
except NameError:  # Kaggle notebook: modules are written to /tmp/src
    sys.path.insert(0, "/tmp/src")
from blocking import FORWARD, NONE, _pairs_evaluated, block_country, texts, token_df  # noqa: E402

K_GRID = [5, 10, 20, 30, 50, 75, 100]
M_GRID = [0, 1, 2, 3, 5, 10]
METHODS = [*FORWARD, "reverse", "rare"]
COLS = ["entity_id", "country", "business_name", "core_name", "city", "addr_norm", "state"]
NON_LATIN = r"[\p{L}&&[^\p{Latin}]]"
T0 = time.time()


def log(msg: str) -> None:
    peak = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20
    print(f"[{time.time() - T0:7.1f}s peak {peak:5.1f} GB] {msg}", flush=True)


def find(root: Path, name: str, parent: str | None = None) -> Path:
    if root.is_file():
        return root
    hits = sorted(p for p in root.rglob(name) if parent is None or p.parent.name == parent)
    if not hits:
        raise FileNotFoundError(f"{name} (parent {parent}) not found under {root}")
    return hits[0]


def md(df: pl.DataFrame, max_rows: int = 60) -> str:
    df = df.head(max_rows)
    fmt = lambda v: f"{v:.4f}" if isinstance(v, float) else str(v)
    rows = [" | ".join(df.columns), " | ".join("---" for _ in df.columns)] + [" | ".join(fmt(v) for v in r) for r in df.iter_rows()]
    return "\n".join(f"| {r} |" for r in rows)


def main() -> None:
    ap = argparse.ArgumentParser()
    ap.add_argument("--norm-dir", default="artifacts/normalised")
    ap.add_argument("--data-dir", default="data_parquet")
    ap.add_argument("--split", default="outputs/eda/g25_split.parquet")
    ap.add_argument("--out-dir", default="artifacts/blocking_eval")
    ap.add_argument("--k-max", type=int, default=max(K_GRID))
    ap.add_argument("--m-max", type=int, default=max(M_GRID))
    a = ap.parse_args()
    out = Path(a.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    norm = lambda sp, k, cols=COLS: pl.read_parquet(find(Path(a.norm_dir), f"{sp}_source{k}.parquet", "normalised"), columns=cols)
    split = pl.read_parquet(find(Path(a.split), "g25_split.parquet"))
    val_ids = split.filter(pl.col("role") == "val")["source1_entity_id"]
    gt = pl.read_parquet(find(Path(a.data_dir), "train_ground_truth.parquet"))
    truth = (
        gt.filter(pl.col("source1_entity_id").is_in(val_ids.implode()))
        .select(pl.col("source1_entity_id").alias("s1"), pl.col("matched_entity_ids").str.split(",").alias("cand"))
        .explode("cand", empty_as_null=True).filter(pl.col("cand").is_not_null() & (pl.col("cand") != ""))
        .with_columns(pl.col("cand").str.slice(0, 2).alias("msrc"))
    )
    tr = {k: norm("train", k) for k in (1, 2, 3)}
    te = {k: norm("test", k, ["country", "core_name", "city", "addr_norm", "state"]) for k in (1, 2, 3)}
    log(f"loaded; {val_ids.len():,} validation S1, {truth.height:,} validation true pairs")

    cands, timing, test_est, attrs = [], [], [], []
    for country in tr[1]["country"].unique().sort().to_list():
        log(f"country {country}")
        s1 = tr[1].filter(pl.col("country") == country)
        pool = pl.concat([tr[2].filter(pl.col("country") == country), tr[3].filter(pl.col("country") == country)])
        test_all = pl.concat([t.filter(pl.col("country") == country) for t in te.values()], how="diagonal_relaxed")
        train_all = pl.concat([s1, pool])
        corpus = {rep: pl.concat([texts(train_all)[rep], texts(test_all)[rep]]) for rep in FORWARD}
        df_counts = token_df(pl.concat([train_all["core_name"], test_all["core_name"]]))
        qmask = s1["entity_id"].is_in(val_ids.implode()).to_numpy()
        long, tim = block_country(s1, pool, qmask, corpus, df_counts, a.k_max, a.m_max, log=lambda m: log(m))
        cands.append(long.with_columns(pl.Series("s1", s1["entity_id"].to_numpy()[long["qi"].to_numpy()]),
                                       pl.Series("cand", pool["entity_id"].to_numpy()[long["pi"].to_numpy()])).drop("qi", "pi"))
        for rep, t in tim.items():
            timing.append({"country": country, "method": rep, **t})
        # test-time cost from the test bucket sizes of this country (same cost model as timing)
        t1 = te[1].filter(pl.col("country") == country)
        tp = pl.concat([te[2].filter(pl.col("country") == country), te[3].filter(pl.col("country") == country)])
        tsb, tpb = t1["state"].fill_null(NONE).to_numpy(), tp["state"].fill_null(NONE).to_numpy()
        test_est.append({"country": country, "test_s1": t1.height, "test_pool": tp.height,
                         "forward_pairs": _pairs_evaluated(tsb, tpb),
                         "reverse_pairs": _pairs_evaluated(tpb[tpb != NONE], tsb, with_none=False)})
        # attributes of validation S1 for group recall
        cnt = pl.concat([train_all["core_name"], test_all["core_name"]]).alias("core_name").value_counts(name="n_core")
        attrs.append(s1.filter(pl.Series(qmask)).select(pl.col("entity_id").alias("s1"), "country", "core_name")
                     .join(cnt, on="core_name", how="left"))
        del s1, pool, test_all, train_all, corpus
        log(f"country {country} done")
    # France etc. (test-only countries) still need a test-time estimate
    for country in sorted(set(te[1]["country"].unique().to_list()) - set(tr[1]["country"].unique().to_list())):
        t1 = te[1].filter(pl.col("country") == country)
        tp = pl.concat([te[2].filter(pl.col("country") == country), te[3].filter(pl.col("country") == country)])
        tsb, tpb = t1["state"].fill_null(NONE).to_numpy(), tp["state"].fill_null(NONE).to_numpy()
        test_est.append({"country": country, "test_s1": t1.height, "test_pool": tp.height,
                         "forward_pairs": _pairs_evaluated(tsb, tpb),
                         "reverse_pairs": _pairs_evaluated(tpb[tpb != NONE], tsb, with_none=False)})

    long = pl.concat(cands)
    wide = long.pivot(on="method", index=["s1", "cand"], values=["rank", "score"], aggregate_function="min")
    for mth in METHODS:
        for c in ("rank", "score"):
            if f"{c}_{mth}" not in wide.columns:
                wide = wide.with_columns(pl.lit(None, dtype=pl.Int64 if c == "rank" else pl.Float32).alias(f"{c}_{mth}"))
    wide = wide.join(truth.select("s1", "cand", pl.lit(True).alias("is_match")), on=["s1", "cand"], how="left").with_columns(
        pl.col("is_match").fill_null(False))
    wide.write_parquet(out / "val_candidates.parquet")
    log(f"candidates: {wide.height:,} (S1, cand) pairs at k={a.k_max}, m={a.m_max}")

    val_s1 = pl.concat(attrs)
    n_s1 = val_s1.group_by("country").len("n_s1")
    t = truth.join(val_s1.select("s1", "country"), on="s1")

    def evaluate(sel: pl.Expr, label: dict) -> list[dict]:
        found = wide.filter(sel).select("s1", "cand")
        hit = t.join(found, on=["s1", "cand"], how="semi").group_by("country").len("hit")
        size = found.join(val_s1.select("s1", "country"), on="s1").group_by("country").len("cands")
        r = (t.group_by("country").len("true").join(hit, on="country", how="left").join(size, on="country", how="left")
             .join(n_s1, on="country").fill_null(0))
        tot = r.select(pl.lit("ALL").alias("country"), *[pl.col(c).sum() for c in ("true", "hit", "cands", "n_s1")])
        r = pl.concat([r.select(tot.columns), tot])
        return [{**label, "country": x["country"], "recall": x["hit"] / max(x["true"], 1),
                 "cands_per_s1": x["cands"] / max(x["n_s1"], 1)} for x in r.iter_rows(named=True)]

    rows = []
    for mth in METHODS:
        grid = M_GRID[1:] if mth == "reverse" else ([10**9] if mth == "rare" else K_GRID)
        for k in grid:
            rows += evaluate(pl.col(f"rank_{mth}") <= k, {"method": mth, "k": k if mth != "rare" else None})
    by_method = pl.DataFrame(rows)
    by_method.write_csv(out / "recall_by_method.csv")

    def union(k: int, m: int, rare: bool = True) -> pl.Expr:
        e = pl.any_horizontal([pl.col(f"rank_{f}") <= k for f in FORWARD])
        if m:
            e = e | (pl.col("rank_reverse") <= m)
        if rare:
            e = e | pl.col("rank_rare").is_not_null()
        return e.fill_null(False)

    rows = []
    for k in K_GRID:
        for m in M_GRID:
            rows += evaluate(union(k, m), {"k": k, "m": m})
    grid = pl.DataFrame(rows)
    grid.write_csv(out / "recall_union_grid.csv")
    log("grid done")

    # recommendation: cheapest (k, m) with overall union recall >= 97% (else the best recall)
    overall = grid.filter(pl.col("country") == "ALL").sort("cands_per_s1")
    ok = overall.filter(pl.col("recall") >= 0.97)
    rec = (ok if ok.height else overall.sort("recall", descending=True)).row(0, named=True)
    k_rec, m_rec = rec["k"], rec["m"]

    # group recall at the recommended config and at k_max/m_max
    matches = pl.concat([tr[2].select("entity_id", "core_name", "business_name", "state"),
                         tr[3].select("entity_id", "core_name", "business_name", "state")]).rename(
        {"entity_id": "cand", "core_name": "m_core", "business_name": "m_name", "state": "m_state"})
    s1_state = tr[1].select(pl.col("entity_id").alias("s1"), pl.col("state").alias("s1_state"))
    g = t.join(val_s1.select("s1", "core_name", "n_core"), on="s1").join(matches, on="cand", how="left").join(s1_state, on="s1")
    g = g.with_columns(pl.Series("tsr", process.cpdist(g["core_name"].to_list(), g["m_core"].fill_null("").to_list(),
                                                       scorer=fuzz.token_set_ratio, workers=-1)))
    groups = {
        "all": pl.lit(True),
        "common name (core_name 5+ in country)": pl.col("n_core") >= 5,
        "low name similarity (tsr < 70)": pl.col("tsr") < 70,
        "Indian-script match record": pl.col("m_name").str.contains(NON_LATIN),
        "state differs (both known)": pl.col("m_state").is_not_null() & (pl.col("m_state") != pl.col("s1_state")),
        "match has no state": pl.col("m_state").is_null(),
    }
    configs = {f"recommended k={k_rec}, m={m_rec}": union(k_rec, m_rec), f"max k={a.k_max}, m={a.m_max}": union(a.k_max, a.m_max),
               **{f"{mth} only (k={a.k_max if mth != 'reverse' else a.m_max})": (pl.col(f"rank_{mth}").is_not_null()) for mth in METHODS}}
    rows = []
    for cname, sel in configs.items():
        found = wide.filter(sel.fill_null(False)).select("s1", "cand", pl.lit(True).alias("found"))
        gg = g.join(found, on=["s1", "cand"], how="left").with_columns(pl.col("found").fill_null(False))
        for gname, cond in groups.items():
            for country, sub in [("ALL", gg.filter(cond))] + [(c, d) for (c,), d in gg.filter(cond).group_by("country")]:
                rows.append({"config": cname, "group": gname, "country": country, "true_pairs": sub.height,
                             "recall": sub["found"].mean() if sub.height else None})
    grp = pl.DataFrame(rows).sort("config", "group", "country")
    grp.write_csv(out / "recall_groups.csv")

    # timing and test-time estimate: seconds per similarity evaluation measured here x test pair counts
    tim = pl.DataFrame(timing)
    tim.write_csv(out / "timing.csv")
    est = pl.DataFrame(test_est)
    rate = {r["method"]: (r["search_s"] / r["pairs_evaluated"]) for r in
            tim.group_by("method").agg(pl.col("search_s").sum(), pl.col("pairs_evaluated").sum()).drop_nulls().iter_rows(named=True)
            if r["pairs_evaluated"]}
    fit_rate = tim.filter(pl.col("fit_transform_s").is_not_null()).select(
        (pl.col("fit_transform_s").sum() / (pl.col("pool").sum() + pl.col("queries").sum()))).item()
    est = est.with_columns(
        *[(pl.col("forward_pairs") * rate.get(f, 0.0) / 60).alias(f"{f}_min") for f in FORWARD],
        (pl.col("reverse_pairs") * rate.get("reverse", 0.0) / 60).alias("reverse_min"),
        ((pl.col("test_s1") + pl.col("test_pool")) * fit_rate * len(FORWARD) / 60).alias("fit_transform_min"),
    ).with_columns(pl.sum_horizontal(pl.col("^.*_min$")).alias("total_min"))
    est.write_csv(out / "test_estimate.csv")

    report = [
        "# Blocking evaluation (validation split vs full train pool per country)\n",
        f"Recommended: **k = {k_rec}, m = {m_rec}**, union recall {rec['recall']:.4f}, {rec['cands_per_s1']:.1f} candidates per S1.\n",
        "## Union recall grid (ALL countries)\n", md(overall.sort("k", "m").drop("country"), 60) + "\n",
        "## Union recall per country at the recommended config\n",
        md(grid.filter((pl.col("k") == k_rec) & (pl.col("m") == m_rec))) + "\n",
        "## Recall per method (ALL countries)\n", md(by_method.filter(pl.col("country") == "ALL").drop("country"), 60) + "\n",
        "## Recall by group\n", md(grp.filter(pl.col("country") == "ALL").drop("country"), 80) + "\n",
        "## Timing on validation (seconds)\n", md(tim) + "\n",
        "## Test-time estimate (minutes, from measured seconds per similarity evaluation)\n", md(est) + "\n",
        f"Peak memory of this run: {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 2**20:.1f} GB.\n",
    ]
    (out / "blocking_report.md").write_text("\n".join(report))
    (out / "config.json").write_text(json.dumps({"k_rec": k_rec, "m_rec": m_rec, "k_grid": K_GRID, "m_grid": M_GRID}, indent=1))
    log(f"done: recommended k={k_rec}, m={m_rec}")


if __name__ == "__main__":
    main()
